# 03 — Benchmark: SGT-QAT Checkpoint as vLLM Drafter

Loads `checkpoints/qwen3-1.7b-sgt-qat/` (exported by notebook 01, PPL 15.91, 68.0%
corrected recovery — see `docs/findings.md`) as the draft model in vLLM's
speculative-decode config (`method="draft_model"`), targeting Qwen3-8B, and measures
it with the same `notebooks/common/bench_utils.py` harness as notebook 02's EAGLE-3
baseline — same low-concurrency (sequential single-request) methodology, so numbers
are directly comparable.

**Checkpoint source**: Google Drive (`MyDrive/sgt-qat-draft-checkpoints/qwen3-1.7b-sgt-qat/`),
not git — see `docs/context.md` "Checkpoint storage" for why. This notebook mounts
Drive and copies it to local disk before loading (don't read straight off the Drive
mount — flaky mid-large-file-read).

**Not yet run.** Depends on notebook 02 having been (re-)run with the current harness
first, so this notebook's Compare step has real no-spec/EAGLE-3 numbers to load.</cell id="cell-0">


## Setup

In [ ]:
import os
REPO_NAME = 'sgt-qat-draft'
if not os.path.isdir(REPO_NAME):
    !git clone https://github.com/Resh19S/sgt-qat-draft.git
%cd {REPO_NAME}
# Existing clones don't auto-update -- pull explicitly so this session has the
# current bench_utils.py etc.
!git pull

!pip install -q vllm
# Needed to decompress the compressed-tensors checkpoint below (transformers uses
# this package's hooks to auto-dequantize on load) -- not guaranteed to already be
# present just because vllm is installed.
!pip install -q compressed-tensors

# Known issue (docs/logs.md 2026-07-23): pip can resolve a CUDA-13-linked vLLM
# binary alongside a CUDA-12.x torch build -- `import vllm` then fails with
# `ImportError: libcudart.so.13`. The cu13 runtime lib is usually already installed
# as a side dependency, just not on the linker's default search path. Fix it here,
# before anything imports torch/vllm, so a fresh session doesn't need a manual
# restart-and-patch cycle.
import glob, subprocess
cu13_libs = glob.glob('/usr/local/lib/python3.*/dist-packages/nvidia/cu13/lib/libcudart.so.13')
if cu13_libs and not os.path.exists('/usr/lib/x86_64-linux-gnu/libcudart.so.13'):
    subprocess.run(['ln', '-sf', cu13_libs[0], '/usr/lib/x86_64-linux-gnu/libcudart.so.13'], check=True)
    subprocess.run(['ldconfig'], check=True)
    print(f"Symlinked {cu13_libs[0]} -> /usr/lib/x86_64-linux-gnu/libcudart.so.13 (CUDA runtime mismatch fix)")

# Re-enabled (2026-07-24, second time): the plain-checkpoint fix resolved the
# original weight_packed ValueError, but loading still crashes with the same
# generic "Engine core initialization failed" wrapper -- a NEW, different
# underlying cause, still hidden by the spawned-subprocess boundary now that
# these were reverted to defaults. Turning them back on to see what THIS crash
# actually is. See docs/logs.md for the full back-and-forth.
os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'
os.environ['VLLM_LOGGING_LEVEL'] = 'DEBUG'

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_CHECKPOINT = Path('/content/drive/MyDrive/sgt-qat-draft-checkpoints/qwen3-1.7b-sgt-qat')
LOCAL_CHECKPOINT = Path('checkpoints/qwen3-1.7b-sgt-qat')

if not LOCAL_CHECKPOINT.exists():
    LOCAL_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    !cp -r {str(DRIVE_CHECKPOINT)} {str(LOCAL_CHECKPOINT)}

# Sanity check: local copy should match the 1.2GB validated in notebook 01. If this
# comes back much smaller, the copy truncated (see docs/logs.md 2026-07-23 -- this
# already happened once due to Drive being full) -- don't proceed to loading it into
# vLLM until this looks right.
!du -sh {str(LOCAL_CHECKPOINT)}

In [ ]:
# CONFIRMED BLOCKER (docs/context.md, docs/findings.md 2026-07-24): vLLM's
# method="draft_model" path (v0.25.1) cannot load our compressed-tensors
# packed checkpoint -- it builds plain unquantized nn.Linear layers and then
# fails to find the expected .weight tensors (checkpoint stores .weight_packed
# instead). Real ValueError, confirmed via full traceback, not a guess.
#
# Fix: reload the checkpoint and DECOMPRESS it, then re-save without
# save_compressed=True. This does NOT require redoing Stage 1/2 -- it's a
# cheap reload-and-resave of the already-exported checkpoint. Sacrifices the
# disk/VRAM compression story for the drafter itself (this becomes a full
# ~3.4GB fp16 checkpoint, not 1.18GB compressed) -- see findings.md for how
# this reframes the memory-footprint comparison.
#
# IMPORTANT (multiple attempts, 2026-07-24):
# - AutoModelForCausalLM.from_pretrained() alone does NOT dequantize.
# - hf_quantizer.dequantize(model) raises NotImplementedError for
#   compressed-tensors in this transformers version.
# - ModelCompressor.from_pretrained_model(model) + .decompress_model(model)
#   DOES correctly unpack weight_packed -> a real, correctly-shaped .weight
#   tensor. But the modules stay a custom quantized module type (not plain
#   nn.Linear), with weight_scale/weight_shape still present via that type's
#   own serialization logic -- deleting from module._buffers didn't work,
#   confirming they're not simple registered buffers. Fix: explicitly REPLACE
#   each such module with a genuine nn.Linear holding just the (already
#   correctly dequantized) weight, discarding the wrapper type entirely.
PLAIN_CHECKPOINT = Path('checkpoints/qwen3-1.7b-sgt-qat-plain')

_QUANT_LEFTOVER_SUBSTRINGS = (
    'weight_packed', 'weight_scale', 'weight_zero_point', 'weight_shape', 'weight_g_idx',
)


def _plain_checkpoint_is_valid(path: Path) -> bool:
    shards = list(path.glob('*.safetensors'))
    if not shards:
        return False
    from safetensors import safe_open
    with safe_open(str(shards[0]), framework='pt') as f:
        keys = list(f.keys())
    return len(keys) > 0 and not any(
        bad in k for k in keys for bad in _QUANT_LEFTOVER_SUBSTRINGS
    )


def _unwrap_quantized_linears(model):
    """Replace every module still carrying quantization metadata (weight_scale
    or weight_shape as an attribute) with a plain nn.Linear holding the same
    (already correctly dequantized) weight -- discards the custom module type
    and whatever custom state_dict() logic it has, rather than fighting it."""
    import torch.nn as nn
    replaced = 0
    for name, module in list(model.named_modules()):
        if hasattr(module, 'weight_scale') or hasattr(module, 'weight_shape'):
            parent_name, _, child_name = name.rpartition('.')
            parent = model.get_submodule(parent_name) if parent_name else model
            weight = module.weight.data
            has_bias = getattr(module, 'bias', None) is not None
            plain = nn.Linear(
                weight.shape[1], weight.shape[0], bias=has_bias, dtype=weight.dtype
            )
            plain.weight.data.copy_(weight)
            if has_bias:
                plain.bias.data.copy_(module.bias.data)
            setattr(parent, child_name, plain)
            replaced += 1
    print(f"Replaced {replaced} quantized modules with plain nn.Linear.")


if not _plain_checkpoint_is_valid(PLAIN_CHECKPOINT):
    import shutil
    shutil.rmtree(PLAIN_CHECKPOINT, ignore_errors=True)  # clear any invalid prior attempt

    from transformers import AutoModelForCausalLM, AutoTokenizer
    from compressed_tensors import ModelCompressor
    import torch as _torch

    _decompress_model = AutoModelForCausalLM.from_pretrained(
        str(LOCAL_CHECKPOINT), dtype=_torch.float16, trust_remote_code=True
    )
    _compressor = ModelCompressor.from_pretrained_model(_decompress_model)
    _compressor.decompress_model(_decompress_model)

    _unwrap_quantized_linears(_decompress_model)

    if getattr(_decompress_model, 'hf_quantizer', None) is not None:
        _decompress_model.hf_quantizer.remove_quantization_config(_decompress_model)
    if hasattr(_decompress_model.config, 'quantization_config'):
        _decompress_model.config.quantization_config = None

    _decompress_tokenizer = AutoTokenizer.from_pretrained(str(LOCAL_CHECKPOINT))
    PLAIN_CHECKPOINT.mkdir(parents=True, exist_ok=True)
    _decompress_model.save_pretrained(str(PLAIN_CHECKPOINT), save_original_format=False)
    _decompress_tokenizer.save_pretrained(str(PLAIN_CHECKPOINT))
    del _decompress_model
    _torch.cuda.empty_cache() if _torch.cuda.is_available() else None

    assert _plain_checkpoint_is_valid(PLAIN_CHECKPOINT), (
        "Decompression still left quantization-related tensors behind even "
        "after replacing the quantized modules with plain nn.Linear. Stop "
        "here and investigate before attempting the (expensive) vLLM load again."
    )
    print("Verified: no quantization-leftover tensors in the decompressed checkpoint.")

!du -sh {str(PLAIN_CHECKPOINT)}  # sanity check: should be ~3.4GB (fp16), not ~1.2GB (compressed)

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path('.').resolve()
sys.path.insert(0, str(REPO_DIR / 'notebooks' / 'common'))

import bench_utils
import torch

assert torch.cuda.is_available(), "No GPU detected."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

In [ ]:
TARGET_MODEL = 'Qwen/Qwen3-8B'
# Points at the PLAIN (decompressed) checkpoint, not the compressed one --
# vLLM's draft_model path can't load compressed-tensors packed weights
# (confirmed ValueError, 2026-07-24, see docs/findings.md). See the
# decompression cell above.
#
# .resolve() to an ABSOLUTE path: a relative path here caused vLLM's
# SpeculativeConfig validation to fail to recognize it as a local directory and
# instead try (and fail, with 401s) to look it up as a Hugging Face repo id.
SGT_QAT_DRAFTER = str(PLAIN_CHECKPOINT.resolve())
NUM_SPECULATIVE_TOKENS = 3   # must match notebook 02's value for a fair comparison
MAX_TOKENS = 256             # must match notebook 02's value
NUM_PROMPTS = 80             # IMPORTANT: set this to whatever notebook 02 actually used --
                              # comparisons are only meaningful if all three conditions
                              # (no-spec, eagle3, draft_model) ran the same prompt count.

# Same placeholder prompts as notebook 02 -- keep identical across notebooks so the
# three conditions are comparable. Still a smoke-test set, not a real benchmark
# dataset (see notebook 02's TODO on swapping to e.g. mt-bench).
PROMPTS = [
    "Explain the difference between speculative decoding and beam search.",
    "Write a short function in Python that reverses a linked list.",
    "Summarize the plot of Pride and Prejudice in three sentences.",
] * (NUM_PROMPTS // 3 + 1)
PROMPTS = PROMPTS[:NUM_PROMPTS]

## Run: SGT-QAT drafter

In [ ]:
speculative_config_sgt_qat = {
    'method': 'draft_model',
    'model': SGT_QAT_DRAFTER,
    'num_speculative_tokens': NUM_SPECULATIVE_TOKENS,
}

# The EAGLE-3 run in notebook 02 already used 38.60GiB of this A100's 40GiB just for
# the target model + a small EAGLE-3 head. Our drafter is a full 1.7B model (much
# bigger than EAGLE-3's lightweight head) loaded alongside the same 8B target --
# real OOM risk on load. vLLM defaults to reserving KV cache sized for
# max_model_len=40960 (Qwen3-8B's full context) regardless of actual usage; capping
# it here frees real memory for the extra model since our prompts/outputs are short.
result_sgt_qat = bench_utils.run_benchmark(
    run_name='sgt_qat_drafter',
    target_model=TARGET_MODEL,
    prompts=PROMPTS,
    speculative_config=speculative_config_sgt_qat,
    max_tokens=MAX_TOKENS,
    llm_kwargs={'max_model_len': 4096},
)
bench_utils.save_result(result_sgt_qat, results_dir=REPO_DIR / 'results')
result_sgt_qat

## Compare against notebook 02's baselines

Loads the most recent `no_spec_decode_*.json` and `baseline_eagle3_*.json` from
`results/` (i.e. whatever notebook 02 was last run with) and puts all three
conditions side by side. Assumes notebook 02 was (re-)run with the current harness
(`gpu_memory_used_bytes` field, sequential single-request methodology) -- if it
wasn't, re-run notebook 02 first or this will error on old-format JSON.

In [ ]:
import json


def _load_latest(results_dir: Path, run_name_prefix: str) -> bench_utils.BenchResult:
    matches = sorted((results_dir).glob(f"{run_name_prefix}_*.json"))
    if not matches:
        raise FileNotFoundError(
            f"No results/{run_name_prefix}_*.json found -- run notebook 02 first."
        )
    latest = matches[-1]
    data = json.loads(latest.read_text())
    return bench_utils.BenchResult(**data)


results_dir = REPO_DIR / 'results'
result_no_spec = _load_latest(results_dir, 'no_spec_decode')
result_eagle3 = _load_latest(results_dir, 'baseline_eagle3')

bench_utils.summarize([result_no_spec, result_eagle3, result_sgt_qat])

print("\nOnce these numbers look sane, transcribe them into docs/findings.md (methods +")
print("numbers) and note anything surprising in docs/logs.md. This is Phase 4 territory")
print("once all three conditions are backed by real (non-placeholder) prompts.")